# Single Subtitle — Generate

> ⚡ **Ligue a GPU** — este notebook roda Whisper. *Ambiente de execução → Alterar o tipo de ambiente de execução → GPU.* Sem ela funciona, mas leva muito mais tempo.

Second step of the pipeline (after `video-base-*.ipynb`): transcribes the narration audio with
Whisper and saves the result as an SRT file on Drive.

This is **not** the "master caption" concept (that will only exist once Language Subtitles is
built, as the segmentation/word template the other languages must follow). Here it's simpler:
this notebook just produces one subtitle file; `caption-single-burn.ipynb` will pick whichever
file `config.nome_legenda_unica` points to (this one, by default) and burn it onto the video.

**Manual correction workflow** (optional, but usually worth doing — Whisper transcriptions
always need a quick pass):
1. Run this notebook — it saves `{NOME}_whisper_{IDIOMA}.srt` to the video's folder on Drive.
2. Download the SRT (last cell), correct it locally (any text editor, or a subtitle editor
   like Subtitle Edit / Aegisub).
3. Upload the corrected file back to Drive, replacing the same file (or saving under a new
   name and pointing `NOME_LEGENDA_UNICA` at it in `caption-single-burn.ipynb`).
4. Run `caption-single-burn.ipynb` whenever you're ready — it always reads whatever is on
   Drive at that moment, never a stale local copy.


In [1]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  📦 SETUP — packages, Google Drive, and modules (run once per session) ║
# ╚══════════════════════════════════════════════════════════════════╝

# ── System packages ──────────────────────────────────────────────────────────
!apt-get -qq -y install ffmpeg > /dev/null 2>&1
print('✅ ffmpeg')

# ── Python packages ───────────────────────────────────────────────────────────
!pip install -q openai-whisper
print('✅ openai-whisper')

# ── Mount Drive (unmount first to avoid a stuck session) ────────────────────
from google.colab import drive
try:
    drive.flush_and_unmount()
except Exception:
    pass
drive.mount('/content/drive', force_remount=True)
print('✅ Drive mounted')

# ── Copy modules from Drive to /content/pipeline ────────────────────────────
import shutil, os, sys, logging
from pathlib import Path

PASTA_DRIVE_RAIZ = "narrated_video"  # fixed for the whole project (same value as Configuration)
PASTA_MODULOS = Path(f"/content/drive/MyDrive/{PASTA_DRIVE_RAIZ}/pipeline/modulos")
DESTINO = Path("/content/pipeline")

if PASTA_MODULOS.exists():
    if DESTINO.exists():
        shutil.rmtree(DESTINO)
    shutil.copytree(PASTA_MODULOS, DESTINO)
    print(f"✅ {len(list(DESTINO.glob('*.py')))} modules copied from {PASTA_MODULOS}")

    # ── A cópia trouxe TODOS os módulos? ───────────────────────────────────────
    # "N módulos copiados" sozinho não quer dizer nada. E o modo de falhar aqui é
    # traiçoeiro: o Drive montado do Colab popula a listagem da pasta com atraso,
    # então um copytree logo depois do mount às vezes enxerga só parte dos
    # arquivos. Já aconteceu de copiar 13 de 31 -- com visto verde -- e o notebook
    # quebrar muito depois, num import, longe da causa.
    #
    # A conferência é de três pontas, porque a causa muda o conserto:
    #   manifesto  o que o repositório tem  (versionado; chega pela cópia)
    #   Drive      o que chegou lá
    #   VM         o que a cópia desta célula trouxe
    # A conferência tem duas perguntas, e SÓ UMA delas precisa do manifesto:
    #
    #   Drive → VM   a cópia acima trouxe tudo?      dá pra ver aqui mesmo
    #   repo → Drive o Drive está em dia?            só o manifesto sabe
    #
    # A versão anterior amarrava as duas ao manifesto: sem ele, imprimia um
    # aviso e seguia SEM CONFERIR NADA. Foi assim que "✅ 13 modules copied"
    # passou com visto verde num Drive que tinha 31 -- justamente no dia em
    # que o manifesto ainda não existia. Comparar 13 com 31 nunca dependeu de
    # manifesto nenhum.
    _no_drive = {f.name for f in PASTA_MODULOS.glob("*.py")}
    _na_vm    = {f.name for f in DESTINO.glob("*.py")}

    # ── Drive → VM ────────────────────────────────────────────────────────
    # O Drive montado do Colab popula a listagem da pasta com atraso, então um
    # copytree logo depois do mount às vezes enxerga só parte dos arquivos.
    # Uma segunda passada, com o mount já quente, costuma resolver.
    _nao_copiados = sorted(_no_drive - _na_vm)
    if _nao_copiados:
        print(f"   ⏳ {len(_nao_copiados)} módulo(s) não vieram na 1ª passada — copiando de novo")
        for _n in _nao_copiados:
            shutil.copyfile(PASTA_MODULOS / _n, DESTINO / _n)
        _na_vm = {f.name for f in DESTINO.glob("*.py")}
        _nao_copiados = sorted(_no_drive - _na_vm)
    if _nao_copiados:
        print(f"\n🚨 {len(_nao_copiados)} módulo(s) estão no Drive mas não copiaram:")
        for _n in _nao_copiados:
            print(f"     {_n}")
        raise SystemExit("Rode ESTA célula de novo — o Drive montado ainda estava acordando.")
    print(f"   ✅ os {len(_no_drive)} módulos do Drive chegaram na VM")

    # ── repositório → Drive ───────────────────────────────────────────────
    _manifesto = PASTA_MODULOS / "_manifesto.txt"
    if not _manifesto.exists():
        print("   ⚠️  sem _manifesto.txt: não dá pra saber se o DRIVE está atrás")
        print("      do repositório. Ele é versionado — rode o repositorio-sincronizar.")
    else:
        _esperados = {l.strip() for l in _manifesto.read_text().splitlines()
                      if l.strip() and not l.startswith("#")}
        _fora_do_drive = sorted(_esperados - _no_drive)
        if _fora_do_drive:
            print(f"\n🚨 {len(_fora_do_drive)} módulo(s) não estão no DRIVE:")
            for _n in _fora_do_drive:
                print(f"     {_n}")
            raise SystemExit("Rode o repositorio-sincronizar.ipynb — o Drive está atrás do repositório.")
        print(f"   ✅ e batem com os {len(_esperados)} do manifesto")

    # ── O Python está segurando a versão anterior? ────────────────────────
    # Copiar arquivo novo por cima não desfaz um import já feito: o Python
    # guarda o módulo em sys.modules e reaproveita. Numa sessão longa, isso
    # faz o notebook rodar com o config.py de ontem mesmo depois de um sync
    # perfeito -- e o sintoma aparece longe da causa (nome de arquivo que
    # mudou, padrão que era pra ter mudado e não mudou). Descarregar aqui
    # equivale a reiniciar o runtime, sem perder o resto da sessão.
    _recarregar = [_n for _n, _m in list(sys.modules.items())
                   if getattr(_m, "__file__", None) and str(DESTINO) in str(_m.__file__)]
    for _n in _recarregar:
        del sys.modules[_n]
    if _recarregar:
        print(f"   ♻️  {len(_recarregar)} módulo(s) já importados foram descarregados —")
        print(f"      o import vai reler a cópia nova (rode as células seguintes de novo)")
else:
    print(f"❌ Modules folder not found: {PASTA_MODULOS}")
    print("   Make sure the .py files are in pipeline/modulos/ on Drive.")

if str(DESTINO) not in sys.path:
    sys.path.insert(0, str(DESTINO))

# ── O ambiente combina com o que este notebook faz? ────────────────────────
# Cota de GPU do Colab é limitada e some sem aviso -- e parte da nossa foi
# gasta em notebook que não usa GPU pra nada, rodando com GPU só porque a
# seleção ficou de antes. Silencioso quando combina.
try:
    from ambiente import avisar_gpu
    avisar_gpu(precisa=True)
except Exception:
    pass

logging.basicConfig(level=logging.INFO, format='%(asctime)s  %(name)-18s  %(levelname)s  %(message)s', datefmt='%H:%M:%S')
os.chdir('/content')
print('✅ Setup complete!')


✅ ffmpeg
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 8.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 7.4 MB/s eta 0:00:00
✅ openai-whisper
Drive not mounted, so nothing to flush and unmount.
Mounted at /content/drive
✅ Drive mounted
✅ 13 modules copied from /content/drive/MyDrive/narrated_video/pipeline/modulos
✅ Setup complete!


In [2]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  ⚙️  CONFIGURATION                                               ║
# ║  ✏️  Edit only this cell — same NOME_ORACAO as video-base-*.ipynb  ║
# ╚══════════════════════════════════════════════════════════════════╝

# ── 1. VIDEO IDENTITY (must match video-base-*.ipynb) ────────────────────────
NOME_ORACAO = "40_Matt_02"

# ── 2. NARRATION LANGUAGE ──────────────────────────────────────────────────
# Language Whisper should transcribe in — must match the actual narration
# audio (e.g. "en" for English, "pt" for Portuguese). Also used to name the
# output file: {NOME_ORACAO}_whisper_{IDIOMA_MESTRE}.srt
IDIOMA_MESTRE = "en"

# ── 3. WHISPER MODEL ────────────────────────────────────────────────────────
# tiny/base = rápido, erra mais · small/medium = mais lento, erra menos.
#
# "small" é o padrão porque texto bíblico é cheio de nome próprio, e é neles
# que o "base" tropeça. Medido no Mateus 2, mesma narração:
#
#     base   24 divergências contra o roteiro   (0,9521)
#            "cheap priests", "and mirror", "sinned out", "was raining"
#     small   ~5 divergências
#
# Como esse SRT vira o mestre de segmentação de TODOS os idiomas, erro aqui
# se propaga pro vídeo inteiro. Ligue a GPU (Ambiente de execução → Alterar
# tipo) e o "small" leva ~1 min; sem GPU, alguns minutos.
MODELO_WHISPER = "small"

# ── 4. DRIVE ROOT FOLDER ──────────────────────────────────────────────────
PASTA_DRIVE_RAIZ = "narrated_video"     # ⚠️ DO NOT CHANGE — fixed for the whole project

# ── CHECK ──────────────────────────────────────────────────────────────────
print("=" * 60)
print("⚙️  CONFIGURATION")
print("=" * 60)
print(f"   Video:           {NOME_ORACAO}")
print(f"   Narration lang:  {IDIOMA_MESTRE}")
print(f"   Whisper model:   {MODELO_WHISPER}")
print(f"   Drive root:      {PASTA_DRIVE_RAIZ}")
print("=" * 60)
print("✅ Configuration ready — proceed to Initialization")


⚙️  CONFIGURATION
   Video:           40_Matt_02
   Narration lang:  en
   Whisper model:   base
   Drive root:      narrated_video
✅ Configuration ready — proceed to Initialization


In [3]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🚀 INITIALIZE PIPELINE                                          ║
# ╚══════════════════════════════════════════════════════════════════╝

import sys
from pathlib import Path

if '/content/pipeline' not in sys.path:
    sys.path.insert(0, '/content/pipeline')

from config import PipelineConfig
from caption_pipeline import CaptionPipeline

config = PipelineConfig(
    NOME_ORACAO      = NOME_ORACAO,
    PASTA_DRIVE_RAIZ = PASTA_DRIVE_RAIZ,
    IDIOMA_MESTRE     = IDIOMA_MESTRE,
)

pipeline = CaptionPipeline(config)

print("=" * 60)
print("✅ PIPELINE INITIALIZED")
print("=" * 60)
print(f"   Video:            {config.NOME_ORACAO}")
print(f"   Folder:           {config.pasta_oracao}")
print(f"   Output filename:  {config.NOME_SRT_PT_WHISPER}")
print("=" * 60)


✅ PIPELINE INITIALIZED
   Video:            40_Matt_02
   Folder:           /content/drive/MyDrive/narrated_video/videos/40_Matt_02
   Output filename:  40_Matt_02_edge_en.srt


In [6]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🎙️ TRANSCRIBE — Whisper over the narration audio                ║
# ║  Saves {NOME}_whisper_{IDIOMA}.srt to the video's folder on Drive.  ║
# ╚══════════════════════════════════════════════════════════════════╝

srt_path = pipeline.transcrever_whisper(modelo=MODELO_WHISPER)
print(f"\n✅ Saved: {srt_path.name}")


100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 153MiB/s]
/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")



✅ Saved: 40_Matt_02_edge_en.srt


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  👀 PREVIEW — quick look at the transcription                    ║
# ╚══════════════════════════════════════════════════════════════════╝

from srt_utils import ler_srt

legendas = ler_srt(srt_path)
print(f"{len(legendas)} caption blocks\n")
for leg in legendas:
    print(f"  [{leg.inicio_str} → {leg.fim_str}]  {leg.texto}")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🔍 CONFERIR — o Whisper contra o roteiro do capítulo             ║
# ║  Lista SÓ onde os dois divergem. Não muda nada.                   ║
# ╚══════════════════════════════════════════════════════════════════╝
#
# O SRT tem duas coisas, e só uma pode estar errada:
#
#     tempos   do Whisper -- ele ouviu o áudio, estão certos
#     texto    do Whisper -- erra nome próprio ("Arkeleus", "Seeing")
#
# O roteiro tem o texto certo (WEB, conferido) e nenhum tempo. Esta célula
# alinha os dois e mostra onde discordam, pra você corrigir só esses pontos.
#
# ⚠️ NÃO substitua tudo pelo roteiro. O David Williams lê ligeiramente
# diferente do escrito em alguns trechos (0,9625 de similaridade, medido no
# biblia-audio-conferir) -- substituir cego trocaria um erro visível, o nome
# errado, por um invisível: legenda dizendo o que o áudio não fala.

import biblia_texto as bt
from srt_utils import ler_srt

CAMINHO_BIBLIA = Path(f"/content/drive/MyDrive/{PASTA_DRIVE_RAIZ}/pipeline/dados_lexico/web-biblia.json")
roteiro_txt = config.pasta_oracao / f"{NOME_ORACAO}_roteiro_versiculos.txt"

if roteiro_txt.exists():
    referencia, origem_ref = roteiro_txt.read_text(encoding="utf-8"), roteiro_txt.name
elif CAMINHO_BIBLIA.exists():
    referencia, origem_ref = bt.roteiro_do_capitulo(NOME_ORACAO, CAMINHO_BIBLIA), "web-biblia.json"
else:
    referencia, origem_ref = "", ""

if not referencia:
    print("⏭️  Sem roteiro pra comparar.")
    print(f"    Nem {roteiro_txt.name} na pasta do vídeo, nem web-biblia.json.")
    print("    Rode o biblia-texto-baixar uma vez, ou a célula de texto do video-base.")
else:
    legendas = ler_srt(srt_path)
    texto_whisper = " ".join(l.texto for l in legendas)
    cmp = bt.comparar(texto_whisper, referencia)

    print(f"📄 referência: {origem_ref}")
    print(f"   Whisper {cmp.palavras_a} palavras · roteiro {cmp.palavras_b} · "
          f"similaridade {cmp.similaridade:.4f}")
    if MODELO_WHISPER in ("tiny", "base"):
        print(f'   ⚠️  MODELO_WHISPER="{MODELO_WHISPER}" erra bastante nome próprio.')
        print('       Boa parte da lista abaixo some com "small" (ligue a GPU).')
    print()

    if cmp.identico:
        print("✅ Idênticos — nada a corrigir.")
    else:
        # Em que bloco cada diferença cai. A posição vem do próprio alinhamento
        # (cmp.posicoes_a), não de procurar o trecho no texto: a primeira versão
        # fazia `find()` do começo do contexto, que é sempre uma palavra comum
        # ("when", "was"), e casava a PRIMEIRA ocorrência -- apontando bloco ~1
        # pra uma diferença que estava no bloco 39. Erro de localização é pior
        # que nenhuma: manda procurar no lugar errado, com confiança.
        limites = []          # (índice da última palavra do bloco, nº do bloco)
        acc = 0
        for i, l in enumerate(legendas, 1):
            acc += len(bt.palavras_comparaveis(l.texto))
            limites.append((acc, i))

        def bloco_de(pos):
            for limite, i in limites:
                if pos < limite:
                    return i
            return len(legendas)

        print(f"⚠️  {len(cmp.diferencas)} trecho(s) divergente(s):\n")
        for (tipo, do_whisper, do_roteiro), pos in zip(cmp.diferencas, cmp.posicoes_a):
            print(f"  bloco {bloco_de(pos)}  ({tipo})")
            print(f"     Whisper: ...{do_whisper}...")
            print(f"     Roteiro: ...{do_roteiro}...")
            print()
        print("Corrija no SRT (célula de download abaixo) só o que for erro de")
        print("transcrição. Onde o Dave leu diferente do escrito, o Whisper está")
        print("certo -- a legenda tem que dizer o que se ouve.")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  📥 DOWNLOAD — SRT (for manual correction)                       ║
# ╚══════════════════════════════════════════════════════════════════╝

from google.colab import files

print(f"📥 Downloading {srt_path.name}...")
files.download(str(srt_path))
print()
print("After correcting it locally, upload it back to Drive at:")
print(f"   {config.pasta_oracao / srt_path.name}")
print("(overwrite the same file — or save under a new name and set")
print(" NOME_LEGENDA_UNICA in caption-single-burn.ipynb to point at it)")
